# 🚢 Titanic Survival Prediction
### Beginner Machine Learning Project

**Goal:** Predict whether a passenger survived the Titanic disaster based on features like age, sex, ticket class, etc.

**Dataset:** [Kaggle Titanic Dataset](https://www.kaggle.com/c/titanic/data) — download `train.csv`

**Algorithm:** Logistic Regression (Binary Classification)

---

## Step 1: Import Libraries

In [ ]:
import pandas as pd               # Data manipulation
import numpy as np                # Numerical operations
import matplotlib.pyplot as plt   # Plotting
import seaborn as sns             # Beautiful visualizations

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

## Step 2: Load the Dataset

> 📥 Download `train.csv` from [Kaggle](https://www.kaggle.com/c/titanic/data) and upload it to Colab or place it in the same folder.

In [ ]:
df = pd.read_csv('train.csv')

print(f'📦 Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Check data types and non-null counts
df.info()

In [ ]:
# Basic statistics for numeric columns
df.describe()

## Step 3: Understand the Columns

| Column | Description |
|--------|-------------|
| `PassengerId` | Just an ID — not useful |
| `Survived` | **Target** → 0 = No, 1 = Yes |
| `Pclass` | Ticket class: 1=Upper, 2=Middle, 3=Lower |
| `Name` | Passenger name |
| `Sex` | male / female |
| `Age` | Age in years |
| `SibSp` | # of siblings/spouses aboard |
| `Parch` | # of parents/children aboard |
| `Ticket` | Ticket number |
| `Fare` | Price paid for ticket |
| `Cabin` | Cabin number (lots of missing values) |
| `Embarked` | Port: C=Cherbourg, Q=Queenstown, S=Southampton |

In [ ]:
print('🎯 Survival counts:')
print(df['Survived'].value_counts())
print(f"\n  0 = Did NOT survive : {df['Survived'].value_counts()[0]}")
print(f"  1 = Survived        : {df['Survived'].value_counts()[1]}")

## Step 4: Exploratory Data Analysis (EDA)
Visualize patterns in the data before building the model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Titanic - Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Overall survival count
sns.countplot(x='Survived', data=df, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Overall Survival Count')
axes[0, 0].set_xticklabels(['Did Not Survive', 'Survived'])

# Survival by Gender
sns.countplot(x='Sex', hue='Survived', data=df, ax=axes[0, 1], palette='Set1')
axes[0, 1].set_title('Survival by Gender')
axes[0, 1].legend(['Did Not Survive', 'Survived'])

# Survival by Passenger Class
sns.countplot(x='Pclass', hue='Survived', data=df, ax=axes[1, 0], palette='Set3')
axes[1, 0].set_title('Survival by Passenger Class')
axes[1, 0].legend(['Did Not Survive', 'Survived'])

# Age Distribution
df['Age'].dropna().hist(ax=axes[1, 1], bins=30, color='steelblue', edgecolor='black')
axes[1, 1].set_title('Age Distribution')
axes[1, 1].set_xlabel('Age')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Survival rate by gender (as percentages)
print('Survival rate by gender:')
print(df.groupby('Sex')['Survived'].mean() * 100)

## Step 5: Handle Missing Values
Check and fill/drop missing data before training.

In [ ]:
print('🔎 Missing values per column:')
print(df.isnull().sum())

In [ ]:
# Age → Fill with MEDIAN (less affected by outliers than mean)
df['Age'].fillna(df['Age'].median(), inplace=True)
print(f"✅ Age: filled with median = {df['Age'].median():.1f}")

# Embarked → Fill with MODE (most common value = 'S')
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
print(f"✅ Embarked: filled with mode = '{df['Embarked'].mode()[0]}'")

# Cabin → Drop entirely (77% missing — too many to fill)
df.drop(columns=['Cabin'], inplace=True)
print('✅ Cabin: dropped (too many missing values)')

print('\n🔎 Missing values after cleaning:')
print(df.isnull().sum())

## Step 6: Feature Selection
Drop columns that are not useful for prediction.

In [ ]:
# PassengerId → just an ID number
# Name        → too unique (each person has a different name)
# Ticket      → too unique/inconsistent
df.drop(columns=['PassengerId', 'Name', 'Ticket'], inplace=True)

print('✅ Dropped: PassengerId, Name, Ticket')
print('\n📋 Remaining columns:', list(df.columns))
df.head()

## Step 7: Encode Categorical Variables
ML models only understand **numbers**, not text like `"male"` or `"S"`.

In [ ]:
le = LabelEncoder()

# Sex: female → 0, male → 1
df['Sex'] = le.fit_transform(df['Sex'])
print('✅ Sex encoded     → female=0, male=1')

# Embarked: C → 0, Q → 1, S → 2
df['Embarked'] = le.fit_transform(df['Embarked'])
print('✅ Embarked encoded → C=0, Q=1, S=2')

print('\n🔍 Dataset after encoding:')
df.head()

## Step 8: Define Features (X) and Target (y)

In [ ]:
# X = input features (everything except Survived)
# y = what we want to predict (Survived)

X = df.drop(columns=['Survived'])
y = df['Survived']

print(f'✅ X (features) shape : {X.shape}')
print(f'✅ y (target)   shape : {y.shape}')
print(f'\n📋 Features used: {list(X.columns)}')

## Step 9: Train / Test Split

- **Training set (80%)** → model learns from this  
- **Test set (20%)** → we evaluate how well it learned  
- `random_state=42` ensures the **same split every time** (reproducibility!)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # Fixed seed → same split every run
)

print(f'✅ Training samples : {X_train.shape[0]}')
print(f'✅ Testing  samples : {X_test.shape[0]}')

## Step 10: Train the Model
We use **Logistic Regression** — great for binary (Yes/No) prediction problems.

In [ ]:
model = LogisticRegression(max_iter=200, random_state=42)
# max_iter=200 → enough iterations for the optimizer to converge

model.fit(X_train, y_train)   # 🚀 Model learns here!

print('✅ Model trained successfully!')

## Step 11: Make Predictions

In [ ]:
y_pred = model.predict(X_test)

print(f'🔮 Predictions (first 10): {y_pred[:10]}')
print(f'🎯 Actual values (first 10): {list(y_test[:10])}')

## Step 12: Evaluate the Model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'{'='*45}')
print(f'  ✅ Accuracy: {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'{'='*45}')

In [ ]:
print('📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

## Step 13: Confusion Matrix

Shows how many predictions were correct vs incorrect:
- **True Negative (TN):** Correctly predicted NOT survived
- **True Positive (TP):** Correctly predicted SURVIVED  
- **False Positive (FP):** Predicted survived but actually didn't
- **False Negative (FN):** Predicted didn't survive but actually did

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Did Not Survive', 'Survived'],
    yticklabels=['Did Not Survive', 'Survived']
)
plt.title('Confusion Matrix', fontweight='bold', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Step 14: Feature Importance
Which features matter most? Logistic Regression assigns a **coefficient** to each feature.  
- **Positive** coefficient → increases survival probability  
- **Negative** coefficient → decreases survival probability  
- **Larger absolute value** → more important feature

In [ ]:
feat_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print('📊 Feature Importance:')
print(feat_importance.to_string(index=False))

plt.figure(figsize=(8, 5))
colors = ['green' if c > 0 else 'red' for c in feat_importance['Coefficient']]
plt.barh(feat_importance['Feature'], feat_importance['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Feature Coefficients\n(Green = increases survival | Red = decreases survival)',
          fontweight='bold')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## Step 15: Predict for a New Passenger
Try predicting survival for a custom passenger!

In [ ]:
# 🧪 Change these values to test different passengers!
# Feature order: Pclass, Sex, Age, SibSp, Parch, Fare, Embarked

new_passenger = pd.DataFrame({
    'Pclass':   [3],       # 1=Upper, 2=Middle, 3=Lower class
    'Sex':      [0],       # 0=female, 1=male
    'Age':      [25],      # Age in years
    'SibSp':    [0],       # Siblings/spouses aboard
    'Parch':    [0],       # Parents/children aboard
    'Fare':     [7.5],     # Ticket fare
    'Embarked': [2]        # 0=Cherbourg, 1=Queenstown, 2=Southampton
})

prediction  = model.predict(new_passenger)[0]
probability = model.predict_proba(new_passenger)[0]

print('🧑 Passenger Details: 3rd class female, age 25, Southampton')
print(f'\n📊 Survival Probability : {probability[1]*100:.1f}%')
print(f'🔮 Prediction           : {"✅ SURVIVED" if prediction == 1 else "❌ DID NOT SURVIVE"}')

---
## 📝 Project Summary

| Item | Details |
|------|---------|
| **Dataset** | Titanic train.csv (891 passengers) |
| **Algorithm** | Logistic Regression |
| **Train/Test Split** | 80% / 20% |
| **Key Feature** | Sex (women had much higher survival rate) |

### 🔑 Key Learnings
- **Missing values** must be handled before training (fill or drop)
- **Label Encoding** converts text categories → numbers
- **`random_state`** in train_test_split ensures reproducibility
- **Logistic Regression** works well for Yes/No prediction problems
- **Feature coefficients** tell us which features matter most

### 🚀 Next Steps to Try
- Try `DecisionTreeClassifier` or `RandomForestClassifier` instead
- Add a new feature: `FamilySize = SibSp + Parch + 1`
- Extract titles from `Name` (Mr., Mrs., Miss.) as a feature
- Use `StandardScaler` to scale features and compare accuracy